# Testing video stream on macbook

The purpose of this notebook is to test the camera stream from the ESP 32 only. Remember to connect to the ESP32 please before using the notebook. 
Will be usefull to use for just collecting stress test and heat samples of the device usage. 
Will be broken up to multiple parts. 

1. Raw video Stream
2. Raw video stream with model
3. Raw video Stream with encryption
4. Raw video Stream with encryption and model
5. Raw video Stream with autoencoder

NOTE: Not all parts of this notebook have been implemented. 
This shows a way to test the stream for future scope. 
As of February 23, 2026 the first 2 have been done. As the work is progress, different parts of the notebooks might fail due to encoding/encryption/model. 

This is just a way to keep everything in one place.


# 1 Raw Video Stream 

# Raw Video with model

In [ ]:
# %% [markdown]
# Live stream classifier + active data collection (single Jupyter cell)

# %%
import os, socket, struct, time, csv, sys
from glob import glob
import cv2
import numpy as np
import torch
import torch.nn.functional as F
from datetime import datetime

# --------------------------
# Config (edit as needed)
# --------------------------
HOST, PORT = "xxx.xxx.x.x", xxxx

# Model discovery
MODEL_DIR = "outputs"           # parent directory with subfolders per run
MODEL_FILE = None               # set a path to override auto-detection, else leave None

# Inference
IMG_SIZE = 600                  # must match your model's expected size
USE_FP16 = False                # enable if your CPU/GPU + model support it

# Active learning / auto-collection
SAVE_ROOT = "dataset/auto_collect"
UNCERTAIN_DIR = "uncertain"     # where we save low-confidence frames
PER_CLASS_DIR = "by_pred"       # optional: save some confident examples per class
CONF_THRESHOLD = 0.80           # below this => uncertain bucket
SAVE_UNCERTAIN = True
SAVE_CONFIDENT_EVERY_N = 0      # set >0 to also save 1 confident frame out of every N per class (e.g., 100)
SAVE_COOLDOWN_SEC = 0.5         # don't save more than once per class/uncertain within this time window
LOG_CSV = os.path.join(SAVE_ROOT, "predictions_log.csv")

# Display
WINDOW_NAME = "ESP32 Stream"
SHOW_PREVIEW = True
FONT = cv2.FONT_HERSHEY_SIMPLEX

# Keyboard shortcuts in the preview window:
#   q = quit
#   s = toggle saving confident samples on/off (if SAVE_CONFIDENT_EVERY_N > 0)
#   u = toggle saving uncertain samples on/off

# --------------------------
# Utilities
# --------------------------
def _now_str():
    return datetime.now().strftime("%Y%m%d_%H%M%S_%f")

def ensure_dir(p):
    os.makedirs(p, exist_ok=True)

def recv_exact(sock, nbytes):
    """Receive exactly nbytes from socket or return None on EOF."""
    buf = bytearray()
    while len(buf) < nbytes:
        pkt = sock.recv(nbytes - len(buf))
        if not pkt:
            return None
        buf += pkt
    return buf

def find_latest_model(model_dir="outputs", model_file_override=None):
    """
    Auto-discovers the newest model + labels pair.
    Accepts either:
      - Manual override path in model_file_override
      - A subdir with either 'resnet18_scripted.pt' or '224img.pt' and a 'labels.txt'
    """
    if model_file_override:
        mf = model_file_override
        if not os.path.exists(mf):
            raise FileNotFoundError(f"MODEL_FILE override not found: {mf}")
        base = os.path.dirname(mf)
        labels_guess = os.path.join(base, "labels.txt")
        if not os.path.exists(labels_guess):
            raise FileNotFoundError(f"labels.txt not found next to {mf}")
        return mf, labels_guess

    if not os.path.exists(model_dir):
        raise FileNotFoundError(f'"{model_dir}" does not exist.')

    # Look at 1st-level subdirectories, collect candidate (pt, labels, mtime)
    candidates = []
    checked = []
    for d in [os.path.join(model_dir, x) for x in os.listdir(model_dir) if os.path.isdir(os.path.join(model_dir, x))]:
        # Accept either filename convention
        pt_candidates = [
            os.path.join(d, "resnet18_scripted.pt"),
            os.path.join(d, "224img.pt"),
        ]
        labels_path = os.path.join(d, "labels.txt")
        for pt_path in pt_candidates:
            checked.append((pt_path, labels_path))
            if os.path.exists(pt_path) and os.path.exists(labels_path):
                candidates.append((pt_path, labels_path, os.path.getmtime(pt_path)))

    if not candidates:
        msg = ["Searched these locations but didn’t find both model and labels:"]
        for pt, lb in checked:
            msg.append(f"  - {pt}  and  {lb}")
        raise FileNotFoundError("\n".join(msg))

    candidates.sort(key=lambda x: x[2], reverse=True)
    pt_path, labels_path, _ = candidates[0]
    return pt_path, labels_path

def load_model_and_labels():
    pt, lb = find_latest_model(MODEL_DIR, MODEL_FILE)
    labels = [x.strip() for x in open(lb, "r", encoding="utf-8").read().splitlines() if x.strip()]
    model = torch.jit.load(pt, map_location="cpu").eval()
    if USE_FP16:
        model = model.half()
    print(f"Loaded model: {pt}  ({len(labels)} classes)")
    return model, labels, pt, lb

def preprocess_bgr(img_bgr, img_size=224, use_fp16=False):
    # BGR -> RGB
    rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    # Resize
    resized = cv2.resize(rgb, (img_size, img_size), interpolation=cv2.INTER_AREA)
    # HWC -> CHW tensor in [0,1]
    x = torch.from_numpy(resized).permute(2, 0, 1).contiguous().float() / 255.0
    # Normalize to ImageNet
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
    x = (x - mean) / std
    if use_fp16:
        x = x.half()
    return x.unsqueeze(0)  # NCHW

def draw_label(img, text, conf):
    s = f"{text}  {conf*100:.1f}%"
    cv2.putText(img, s, (16, 38), FONT, 1.0, (0, 255, 0), 2, cv2.LINE_AA)

def init_logging(csv_path):
    ensure_dir(os.path.dirname(csv_path))
    new_file = not os.path.exists(csv_path)
    f = open(csv_path, "a", newline="", encoding="utf-8")
    w = csv.writer(f)
    if new_file:
        w.writerow(["timestamp", "model_path", "pred_label", "pred_index", "confidence", "saved_path"])
    return f, w

# --------------------------
# Setup
# --------------------------
model, labels, model_path, labels_path = load_model_and_labels()
num_classes = len(labels)

ensure_dir(SAVE_ROOT)
uncertain_path = os.path.join(SAVE_ROOT, UNCERTAIN_DIR)
ensure_dir(uncertain_path)
per_class_root = os.path.join(SAVE_ROOT, PER_CLASS_DIR)
ensure_dir(per_class_root)

# per-class directories and counters
class_dirs = {}
class_counts = {}
last_saved_time = {"__uncertain__": 0.0}
for i, lb in enumerate(labels):
    d = os.path.join(per_class_root, lb)
    ensure_dir(d)
    class_dirs[i] = d
    class_counts[i] = 0
    last_saved_time[i] = 0.0

log_f, log_w = init_logging(LOG_CSV)

save_uncertain = SAVE_UNCERTAIN
save_confident_every_n = SAVE_CONFIDENT_EVERY_N

# --------------------------
# Connect socket
# --------------------------
print(f"Connecting to {HOST}:{PORT} ...")
s = socket.socket()
s.setsockopt(socket.IPPROTO_TCP, socket.TCP_NODELAY, 1)
s.connect((HOST, PORT))
print("Connected.")

if SHOW_PREVIEW:
    cv2.namedWindow(WINDOW_NAME, cv2.WINDOW_NORMAL)

# --------------------------
# Main loop
# --------------------------
try:
    while True:
        # read frame header (4 bytes, big-endian length)
        hdr = recv_exact(s, 4)
        if not hdr:
            print("Stream ended.")
            break
        frame_len = struct.unpack(">I", hdr)[0]
        jpg = recv_exact(s, frame_len)
        if not jpg:
            print("Stream ended mid-frame.")
            break

        img = cv2.imdecode(np.frombuffer(jpg, np.uint8), cv2.IMREAD_COLOR)
        if img is None:
            continue

        # Inference
        x = preprocess_bgr(img, IMG_SIZE, USE_FP16)
        with torch.no_grad():
            logits = model(x)
            probs = F.softmax(logits, dim=1)
            conf, pred_idx = torch.max(probs, dim=1)
            conf = float(conf.item())
            pred_idx = int(pred_idx.item())
        pred_label = labels[pred_idx].upper()

        # Print to console
        print(pred_label)

        # Overlay on preview
        if SHOW_PREVIEW:
            vis = img.copy()
            draw_label(vis, pred_label, conf)
            cv2.imshow(WINDOW_NAME, vis)
            key = cv2.waitKey(1) & 0xFF
            if key == ord('q'):
                print("Quit requested.")
                break
            elif key == ord('u'):
                save_uncertain = not save_uncertain
                print(f"[toggle] save_uncertain = {save_uncertain}")
            elif key == ord('s'):
                if save_confident_every_n > 0:
                    save_confident_every_n = 0
                else:
                    save_confident_every_n = SAVE_CONFIDENT_EVERY_N if SAVE_CONFIDENT_EVERY_N > 0 else 50
                print(f"[toggle] save_confident_every_n = {save_confident_every_n}")

        # Decide saving
        saved_path = ""
        ts = time.time()
        ts_str = _now_str()

        if conf < CONF_THRESHOLD and save_uncertain:
            # throttle
            if ts - last_saved_time["__uncertain__"] >= SAVE_COOLDOWN_SEC:
                fname = f"{ts_str}_pred-{pred_label}_conf-{conf:.3f}.jpg"
                saved_path = os.path.join(uncertain_path, fname)
                cv2.imwrite(saved_path, img)
                last_saved_time["__uncertain__"] = ts

        elif save_confident_every_n > 0:
            class_counts[pred_idx] += 1
            if class_counts[pred_idx] % save_confident_every_n == 0:
                # throttle
                if ts - last_saved_time[pred_idx] >= SAVE_COOLDOWN_SEC:
                    fname = f"{ts_str}_{labels[pred_idx]}_conf-{conf:.3f}.jpg"
                    saved_path = os.path.join(class_dirs[pred_idx], fname)
                    cv2.imwrite(saved_path, img)
                    last_saved_time[pred_idx] = ts

        # Log row
        log_w.writerow([ts_str, model_path, labels[pred_idx], pred_idx, f"{conf:.6f}", saved_path])
        log_f.flush()

except KeyboardInterrupt:
    print("Stopped by user.")
except Exception as e:
    print(f"Error: {e}")
finally:
    try:
        s.close()
    except Exception:
        pass
    try:
        if SHOW_PREVIEW:
            cv2.destroyAllWindows()
    except Exception:
        pass
    try:
        log_f.close()
    except Exception:
        pass
